# Baseline centralizado — almendros

Este notebook entrena MobileNetV2 con **todos los datos combinados** (4 contextos juntos, sin federación) como **techo de referencia** para comparar con los resultados de Federated Learning.

**Diseño experimental** (justificación para la memoria):
- **Mismo modelo** que el FL: `Net(phase, unfreeze_last_n_layers)` de `almendros_fl.task`.
- **Mismos transforms**: train con augmentation, val sin augmentation.
- **Mismo split 80/20 con seed=42** por contexto, luego se combinan los splits → train_combined y val_combined.
- **3 seeds (42, 123, 2024)** para reportar media ± std.
- **Pipeline en 2 fases**: Fase 1 (frozen) → Fase 2 (fine-tuning) encadenadas, igual que FL.
- **Early stopping**: paciencia 3, monitor val_accuracy.

El val_combined que se usa aquí es **EXACTAMENTE el mismo** conjunto de imágenes que `load_centralized_test()` en `task.py`, así que las accuracies son directamente comparables con `global_accuracy` del FL.

## 1. Imports y configuración

In [1]:
import sys
from pathlib import Path

# Asegurar que podemos importar el paquete almendros_fl
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import copy
import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt

# Importar TODO desde task.py para garantizar consistencia con FL
from almendros_fl.task import (
    Net,
    CONTEXTS,
    DATA_ROOT,
    NUM_CLASSES,
    CLASS_NAMES,
    TransformedSubset,
    _train_transform,
    _eval_transform,
    set_seed,
    test,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Contextos: {CONTEXTS}")
print(f"Clases: {CLASS_NAMES}")

PyTorch: 2.4.0+cu121
CUDA disponible: False
DATA_ROOT: /home/solano/TFM/Flower_ejemplo/almendros/almendros_fl/data
Contextos: ['manana', 'tarde', 'nublado', 'otros_moviles']
Clases: ['FOTOS_ALMENDRO_SANO', 'FOTOS_MALVAS', 'FOTOS_ORUGAS_BLANCAS', 'FOTOS_VALLICO']


## 2. Carga de datos centralizada

Replicamos exactamente la lógica de `load_data` y `load_centralized_test`, pero combinando los 4 contextos en un único dataset de train y otro de val.

In [2]:
def load_centralized_data(batch_size: int = 32, split_seed: int = 42):
    """Combina los 4 contextos en train_combined y val_combined.

    Para CADA contexto:
      1. Carga ImageFolder sin transform.
      2. Hace random_split 80/20 con seed=42 (igual que load_data en FL).
      3. Envuelve cada split con su transform correspondiente.

    Luego concatena los 4 train y los 4 val.

    Returns:
        train_loader, val_loader, n_train, n_val
    """
    train_datasets = []
    val_datasets = []

    for context_name in CONTEXTS:
        ctx_dir = Path(DATA_ROOT) / context_name
        if not ctx_dir.exists():
            raise FileNotFoundError(f"Falta {ctx_dir}")

        base_dataset = ImageFolder(root=str(ctx_dir), transform=None)
        n_total = len(base_dataset)
        n_train = int(0.8 * n_total)
        n_val = n_total - n_train

        # IMPORTANTE: misma seed=42 que load_data → mismo split exacto
        generator = torch.Generator().manual_seed(split_seed)
        train_sub, val_sub = random_split(
            base_dataset, [n_train, n_val], generator=generator
        )

        train_datasets.append(TransformedSubset(train_sub, _train_transform()))
        val_datasets.append(TransformedSubset(val_sub, _eval_transform()))

    train_combined = ConcatDataset(train_datasets)
    val_combined = ConcatDataset(val_datasets)

    train_loader = DataLoader(train_combined, batch_size=batch_size,
                              shuffle=True, num_workers=0)
    val_loader = DataLoader(val_combined, batch_size=batch_size,
                            shuffle=False, num_workers=0)

    return train_loader, val_loader, len(train_combined), len(val_combined)


# Test rápido
train_loader, val_loader, n_train, n_val = load_centralized_data()
print(f"Train combinado: {n_train} imgs")
print(f"Val combinado:   {n_val} imgs")
print(f"Total:           {n_train + n_val} imgs")
print()
imgs, labels = next(iter(train_loader))
print(f"Batch train: imgs={imgs.shape}, labels={labels.shape}")

Train combinado: 1600 imgs
Val combinado:   400 imgs
Total:           2000 imgs

Batch train: imgs=torch.Size([32, 3, 128, 128]), labels=torch.Size([32])


## 3. Funciones de entrenamiento

Implementamos un `train_epoch` clásico (sin término proximal, sin FedBN — somos centralizados), un `evaluate` que usa el `test()` ya existente, y un `early_stopping` simple.

In [3]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Una época de entrenamiento. Devuelve loss promedio."""
    model.train()
    total_loss = 0.0
    total_examples = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        total_examples += images.size(0)
    return total_loss / total_examples


@dataclass
class EarlyStopping:
    """Early stopping con tracking del best model."""
    patience: int = 3
    monitor: str = "val_accuracy"
    best_value: float = -1.0
    best_epoch: int = 0
    counter: int = 0
    triggered: bool = False
    best_state_dict: dict | None = None

    def step(self, value: float, epoch: int, state_dict: dict) -> bool:
        """Devuelve True si hay que parar."""
        if value > self.best_value:
            self.best_value = value
            self.best_epoch = epoch
            self.best_state_dict = {k: v.detach().clone()
                                    for k, v in state_dict.items()}
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
                return True
            return False


def train_phase(model, train_loader, val_loader, lr: float,
                max_epochs: int, patience: int, device,
                phase_name: str = "Fase"):
    """Entrena hasta max_epochs o hasta que early stopping pare.

    Returns:
        dict con: best_val_acc, best_val_loss, best_epoch, n_epochs_run,
                  early_stop_triggered, history (lista de dicts por época).
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )

    es = EarlyStopping(patience=patience)
    history = []

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = test(model, val_loader, device)
        epoch_time = time.time() - t0

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "time_s": epoch_time,
        })

        stop = es.step(val_acc, epoch, model.state_dict())
        marker = " (nuevo best)" if es.best_epoch == epoch else ""
        print(f"  [{phase_name} ep{epoch:3d}] "
              f"train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | "
              f"val_acc={val_acc:.4f}{marker} | "
              f"{epoch_time:.1f}s")

        if stop:
            print(f"  [{phase_name}] Early stopping en epoca {epoch}. "
                  f"Best ep {es.best_epoch} con val_acc={es.best_value:.4f}")
            break

    # Restaurar mejor modelo
    if es.best_state_dict is not None:
        model.load_state_dict(es.best_state_dict)

    # Metricas finales con el best
    final_val_loss, final_val_acc = test(model, val_loader, device)

    return {
        "best_val_acc": es.best_value,
        "best_val_loss": final_val_loss,
        "best_epoch": es.best_epoch,
        "n_epochs_run": len(history),
        "early_stop_triggered": es.triggered,
        "history": history,
    }

## 4. Pipeline completo: Fase 1 → Fase 2

Replicamos exactamente el flujo del FL:

1. **Fase 1**: MobileNetV2 con base congelada, solo cabeza Dense entrenable.
2. **Fase 2**: descongelar últimas 30 capas, LR bajo (1e-5).

In [4]:
def run_centralized_pipeline(seed: int, batch_size: int = 32,
                              max_epochs_phase1: int = 50,
                              max_epochs_phase2: int = 30,
                              patience: int = 3,
                              lr_phase1: float = 1e-3,
                              lr_phase2: float = 1e-5,
                              unfreeze_n: int = 30):
    """Ejecuta Fase 1 + Fase 2 encadenadas con una seed.

    Returns:
        dict con resultados de ambas fases.
    """
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, val_loader, n_train, n_val = load_centralized_data(
        batch_size=batch_size
    )

    print(f"\n{'=' * 70}")
    print(f"  SEED = {seed}  |  device = {device}")
    print(f"  Train: {n_train} | Val: {n_val}")
    print(f"{'=' * 70}")

    # FASE 1: base congelada
    print(f"\n-- FASE 1: base congelada, lr={lr_phase1}")
    model = Net(phase=1, unfreeze_last_n_layers=unfreeze_n).to(device)
    res_phase1 = train_phase(
        model, train_loader, val_loader,
        lr=lr_phase1, max_epochs=max_epochs_phase1,
        patience=patience, device=device, phase_name="P1",
    )

    # FASE 2: fine-tuning
    print(f"\n-- FASE 2: descongelar ultimas {unfreeze_n} capas, lr={lr_phase2}")
    model.phase = 2
    model._configure_phase()
    res_phase2 = train_phase(
        model, train_loader, val_loader,
        lr=lr_phase2, max_epochs=max_epochs_phase2,
        patience=patience, device=device, phase_name="P2",
    )

    return {
        "seed": seed,
        "phase1": res_phase1,
        "phase2": res_phase2,
    }

## 5. Ejecutar las 3 seeds

**Esto tardara tiempo.** En CPU, calcula ~30-60 min por seed (Fase 1 corta + Fase 2 mas larga). En total ~1.5-3h para las 3 seeds.

Si quieres ir mas rapido para una primera prueba, baja `max_epochs_phase1` y `max_epochs_phase2` a 5 cada uno (pasandolos como argumento).

In [5]:
SEEDS = [42, 123, 2024]
results = []

for seed in SEEDS:
    res = run_centralized_pipeline(seed=seed)
    results.append(res)

print("\n\n" + "=" * 70)
print(f"  COMPLETADAS {len(results)} SEEDS")
print("=" * 70)


  SEED = 42  |  device = cpu
  Train: 1600 | Val: 400

-- FASE 1: base congelada, lr=0.001
  [P1 ep  1] train_loss=0.8459 | val_loss=0.5316 | val_acc=0.7950 (nuevo best) | 165.8s
  [P1 ep  2] train_loss=0.5345 | val_loss=0.4121 | val_acc=0.8200 (nuevo best) | 147.3s
  [P1 ep  3] train_loss=0.4705 | val_loss=0.3632 | val_acc=0.8550 (nuevo best) | 147.4s
  [P1 ep  4] train_loss=0.4507 | val_loss=0.3609 | val_acc=0.8425 | 143.5s
  [P1 ep  5] train_loss=0.4359 | val_loss=0.3291 | val_acc=0.8625 (nuevo best) | 142.4s
  [P1 ep  6] train_loss=0.4098 | val_loss=0.3547 | val_acc=0.8475 | 144.5s
  [P1 ep  7] train_loss=0.3881 | val_loss=0.4423 | val_acc=0.8125 | 142.4s
  [P1 ep  8] train_loss=0.3770 | val_loss=0.3254 | val_acc=0.8600 | 144.1s
  [P1] Early stopping en epoca 8. Best ep 5 con val_acc=0.8625

-- FASE 2: descongelar ultimas 30 capas, lr=1e-05
  [P2 ep  1] train_loss=0.3881 | val_loss=0.3152 | val_acc=0.8625 (nuevo best) | 143.8s
  [P2 ep  2] train_loss=0.3363 | val_loss=0.3022 | val

## 6. Resumen y guardado de resultados

In [6]:
# Construir DataFrame con resultados por seed y fase
rows = []
for r in results:
    rows.append({
        "seed": r["seed"],
        "phase": 1,
        "best_val_acc": r["phase1"]["best_val_acc"],
        "best_val_loss": r["phase1"]["best_val_loss"],
        "best_epoch": r["phase1"]["best_epoch"],
        "n_epochs_run": r["phase1"]["n_epochs_run"],
        "early_stop_triggered": r["phase1"]["early_stop_triggered"],
    })
    rows.append({
        "seed": r["seed"],
        "phase": 2,
        "best_val_acc": r["phase2"]["best_val_acc"],
        "best_val_loss": r["phase2"]["best_val_loss"],
        "best_epoch": r["phase2"]["best_epoch"],
        "n_epochs_run": r["phase2"]["n_epochs_run"],
        "early_stop_triggered": r["phase2"]["early_stop_triggered"],
    })

df = pd.DataFrame(rows)
print("Detalle por seed:")
print(df.to_string(index=False))

# Agregado por fase
agg = df.groupby("phase").agg(
    acc_mean=("best_val_acc", "mean"),
    acc_std=("best_val_acc", "std"),
    loss_mean=("best_val_loss", "mean"),
    loss_std=("best_val_loss", "std"),
    epochs_mean=("n_epochs_run", "mean"),
).reset_index()

print("\nResumen (media +/- std):")
print(agg.to_string(index=False))

# Guardar a CSV
output_dir = Path("results") / "_centralized_baseline"
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "centralized_detail.csv", index=False)
agg.to_csv(output_dir / "centralized_summary.csv", index=False)

# Guardar JSON completo con history
with open(output_dir / "centralized_full.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"\nGuardado en: {output_dir}/")

Detalle por seed:
 seed  phase  best_val_acc  best_val_loss  best_epoch  n_epochs_run  early_stop_triggered
   42      1        0.8625       0.329077           5             8                  True
   42      2        0.9050       0.260354           8            11                  True
  123      1        0.8850       0.301725           5             8                  True
  123      2        0.9150       0.228915          10            13                  True
 2024      1        0.8725       0.339220           3             6                  True
 2024      2        0.9075       0.247062          10            13                  True

Resumen (media +/- std):
 phase  acc_mean  acc_std  loss_mean  loss_std  epochs_mean
     1  0.873333 0.011273   0.323341  0.019394     7.333333
     2  0.909167 0.005204   0.245444  0.015782    12.333333

Guardado en: results/_centralized_baseline/


## 7. Comparativa con FL (tabla + grafica)

Para esta celda necesitamos los resultados de los runs federados. Asumimos que has ejecutado la tanda nocturna (`./run_all_6.sh`) y luego `python aggregate_seeds.py`, que produce `results/summary_multiseed.csv`.

Si todavia no has lanzado la tanda federada, esta celda dara error: no pasa nada, vuelve cuando los tengas.

In [7]:
# Cargar resultados FL agregados
fl_summary_path = Path("results") / "summary_multiseed.csv"

if not fl_summary_path.exists():
    print(f"AVISO: No existe {fl_summary_path}.")
    print("Ejecuta la tanda federada y aggregate_seeds.py primero.")
    fl_df = None
else:
    fl_df = pd.read_csv(fl_summary_path)
    print("Resultados FL:")
    print(fl_df.to_string(index=False))

AVISO: No existe results/summary_multiseed.csv.
Ejecuta la tanda federada y aggregate_seeds.py primero.


In [8]:
# Construir tabla comparativa
if fl_df is not None:
    fl_rows = []
    for _, row in fl_df.iterrows():
        fl_rows.append({
            "method": row["strategy"],
            "phase": row["phase"],
            "acc_mean": row["acc_mean"],
            "acc_std": row["acc_std"],
        })

    for _, row in agg.iterrows():
        fl_rows.append({
            "method": "Centralizado",
            "phase": row["phase"],
            "acc_mean": row["acc_mean"],
            "acc_std": row["acc_std"],
        })

    comparison = pd.DataFrame(fl_rows)
    comparison["acc_str"] = comparison.apply(
        lambda r: f"{r['acc_mean']:.4f} +/- {r['acc_std']:.4f}", axis=1
    )

    table = comparison.pivot(index="method", columns="phase", values="acc_str")
    print("Tabla comparativa (accuracy media +/- std):")
    print(table.to_string())
    table.to_csv(output_dir / "fl_vs_centralized.csv")
else:
    print("Saltando tabla comparativa (faltan resultados FL).")

Saltando tabla comparativa (faltan resultados FL).


In [9]:
# Grafica de barras con error bars
if fl_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, phase in zip(axes, [1, 2]):
        sub = comparison[comparison["phase"] == phase].copy()
        sub = sub.sort_values("acc_mean", ascending=True)

        colors = ["#2E86AB" if m != "Centralizado" else "#E63946"
                  for m in sub["method"]]

        ax.barh(sub["method"], sub["acc_mean"], xerr=sub["acc_std"],
                color=colors, capsize=4, edgecolor="black", linewidth=0.5)
        ax.set_xlabel("Accuracy global (media +/- std)")
        ax.set_title(f"Fase {phase}")
        ax.set_xlim(0, 1.0)
        ax.grid(axis="x", alpha=0.3)

        for i, (mean, std) in enumerate(zip(sub["acc_mean"], sub["acc_std"])):
            ax.text(mean + std + 0.01, i, f"{mean:.3f}",
                    va="center", fontsize=9)

    fig.suptitle("Federated Learning vs. Baseline Centralizado",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(output_dir / "fl_vs_centralized.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"\nGrafica guardada en: {output_dir / 'fl_vs_centralized.png'}")
else:
    print("Saltando grafica (faltan resultados FL).")

Saltando grafica (faltan resultados FL).


## 8. Conclusiones para la memoria

Plantilla para escribir en la memoria/discusion del TFM, una vez tengas ambos resultados:

> El baseline centralizado, entrenado con los 4 contextos combinados y con la **misma arquitectura, transforms y splits** que el experimento federado, alcanza una accuracy global de **X.XXXX +/- Y.YYYY** en Fase 2. Comparado con la mejor estrategia federada (FedXX, **A.AAAA +/- B.BBBB**), el **gap de federacion** es de **Z puntos porcentuales**.
>
> Esto demuestra que [interpretacion: el FL preserva la capacidad del modelo / el FL pierde poco / el FL pierde mucho]. La existencia de este techo de referencia permite contextualizar la accuracy obtenida y descartar que el dataset tenga un limite superior muy diferente al alcanzado por el FL.

Sustituye los placeholders con tus valores reales.